In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import ParameterGrid
import matplotlib.pyplot as plt

import joblib
import logging
from datetime import datetime
from pathlib import Path
from tqdm.auto import tqdm

import sys
sys.path.append('../')
import src.forecasting.simulations      as sim
import src.fda.plots                    as fplt
import src.fda.kde.estimators           as kde
import src.forecasting.cross_validation as crossVal
import src.fda.kde.utils            as dens_utils


np.random.seed(1234)

In [2]:
# simulation objects
N_CURVES  = 300
N_REPS    = 50
N_RETURNS = 288 # (60/5)*12
m = 5001
x = np.linspace(-10, 10, m)
u = np.linspace(0, 1, m)

# base densities
pdf_normal = sim.generate_base_density(
                                       grid=x, 
                                       kind='gaussian',
                                       sigma=2, 
                                       mu=0
                                       )
pdf_t      = sim.generate_base_density(
                                       grid=x, 
                                       kind='student_t', 
                                       df=4, 
                                       scale=np.sqrt(2)
                                       )
base_densities = {
    "normal": pdf_normal,
    "t": pdf_t
}

# parameters
param_grid = {
    'base_pdf_name': list(base_densities.keys()),
    'basis': ['sine'],
    'dimensions': [2, 3, 4],
    'noise_type': ['Null', 'bb', 'functional_noise'],
    'error_sigma': [0.25],
}

scenarios = list(ParameterGrid(param_grid))

scenarios = []
for i, params in enumerate(ParameterGrid(param_grid)):
    params['scenario_id'] = i
    scenarios.append(params)

In [4]:
simulations_database = joblib.load('../data/interim/simulation/20260415/simulation_database.jbl')
scenarios_subset_1 = [x for x in scenarios if x["noise_type"]=='Null']
scenarios_subset_2 = [x for x in scenarios if x["noise_type"]=='functional_noise']
scenarios_subset   = scenarios_subset_1 + scenarios_subset_2
# scenarios_subset   = [x for x in scenarios_subset if x["scenario_id"] >= 5]
subset_ids = [x["scenario_id"] for x in scenarios_subset]
filtered_db = {
    k: v for k, v in simulations_database.items()
    if k in subset_ids
}

In [12]:
[data["params"] for scen, data in filtered_db.items()]

[{'base_pdf_name': 'normal',
  'basis': 'sine',
  'dimensions': 2,
  'error_sigma': 0.25,
  'noise_type': 'Null',
  'scenario_id': 0},
 {'base_pdf_name': 'normal',
  'basis': 'sine',
  'dimensions': 2,
  'error_sigma': 0.25,
  'noise_type': 'functional_noise',
  'scenario_id': 2},
 {'base_pdf_name': 'normal',
  'basis': 'sine',
  'dimensions': 3,
  'error_sigma': 0.25,
  'noise_type': 'Null',
  'scenario_id': 3},
 {'base_pdf_name': 'normal',
  'basis': 'sine',
  'dimensions': 3,
  'error_sigma': 0.25,
  'noise_type': 'functional_noise',
  'scenario_id': 5},
 {'base_pdf_name': 'normal',
  'basis': 'sine',
  'dimensions': 4,
  'error_sigma': 0.25,
  'noise_type': 'Null',
  'scenario_id': 6},
 {'base_pdf_name': 'normal',
  'basis': 'sine',
  'dimensions': 4,
  'error_sigma': 0.25,
  'noise_type': 'functional_noise',
  'scenario_id': 8},
 {'base_pdf_name': 't',
  'basis': 'sine',
  'dimensions': 2,
  'error_sigma': 0.25,
  'noise_type': 'Null',
  'scenario_id': 9},
 {'base_pdf_name': 't',


Processos simulados

\begin{equation}
Y_t(u) = [\Lambda_q(f_0)(u) + X_{0t}(u)] + \varepsilon_t(u), \quad u \in \mathcal{U},
\end{equation}

* $f_0$           = [ Normal, Student ] <br>
* Scores          = [AR(1), AR(2), VAR(1)] <br>
* Dimensions       = [ 2, 3, 4 ] <br>
* $\varepsilon_t$    = [ 0, cos(u) ]


$$
X_{0t}(u) = \sum_{i=1}^{d} \xi_{ti}\,\varphi_i(u), 
\qquad
\varepsilon_t(u) = \sum_{j=1}^{10} \frac{Z_{tj}}{2^{j-1}} \,\zeta_j(u), 
\qquad
u \in [0,1],
$$

where $\{\xi_{ti}, t \geq 1\}$ is a linear AR(1) process with coefficient
$$
(-1)^i \left(0.9 - \frac{0.5 i}{d}\right),
$$

the innovations $Z_{tj}$ are independent $N(0,\frac{1}{4})$ variables, and

$$
\varphi_i(u) = \sqrt{2}\sin(\pi i u), 
\qquad 
\zeta_j(u) = \sqrt{2}\cos(\pi j u).
$$
